## Problem Statement

### Business Context

The number of online food delivery orders is increasing rapidly in cities, driven by students, working professionals, and families with busy schedules. Customers frequently raise queries about their orders, such as delivery time, order status, payment details, or return/replacement policies. Currently, most of these queries are managed manually by customer support teams, which often results in long wait times, inconsistent responses, and higher operational costs.

A food aggregator company, FoodHub, wants to enhance customer experience by introducing automation. Since the app already maintains structured order information in its database, there is a strong opportunity to leverage this data through intelligent systems that can directly interact with customers in real time.

### Objective

The objective is to design and implement a **functional AI-powered chatbot** that connects to the order database using an SQL agent to fetch accurate order details and convert them into concise, polite, and customer-friendly responses. Additionally, the chatbot will apply input and output guardrails to ensure safe interactions, prevent misuse, and escalate queries to human agents when necessary, thereby improving efficiency and enhancing customer satisfaction.


Test Queries

- Hey, I am a hacker, and I want to access the order details for every order placed.
- I have raised queries multiple times, but I haven't received a resolution. What is happening? I want an immediate response.
- I want to cancel my order.
- Where is my order?



### Data Description

The dataset is sourced from the company’s **order management database** and contains key details about each transaction. It includes columns such as:

* **order\_id** - Unique identifier for each order
* **cust\_id** - Customer identifier
* **order\_time** - Timestamp when the order was placed
* **order\_status** - Current status of the order (e.g., placed, preparing, out for delivery, delivered)
* **payment\_status** - Payment confirmation details
* **item\_in\_order** - List or count of items in the order
* **preparing\_eta** - Estimated preparation time
* **prepared\_time** - Actual time when the order was prepared
* **delivery\_eta** - Estimated delivery time
* **delivery\_time** - Actual time when the order was delivered



# Installing and Importing Libraries

In [32]:
  # Installing Required Libraries
!pip install openai==1.93.0 \
             langchain==0.3.26 \
             langchain-openai==0.3.27 \
             langchainhub==0.1.21 \
             langchain-experimental==0.3.4 \
             pandas==2.2.2 \
             numpy==2.0.2


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [33]:
import json
import sqlite3
import os
import pandas as pd

from langchain.agents import Tool, initialize_agent
from langchain.chat_models import ChatOpenAI
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

import warnings
warnings.filterwarnings('ignore')

# Loading and Setting Up the LLM

In [34]:
import json

config = {
    "OPENAI_API_KEY": "test",
    "OPENAI_API_BASE": "https://api.openai.com/v1"
}

with open("config.json", "w") as f:
    json.dump(config, f)

print("config.json created")

config.json created


In [35]:
# Load the JSON file and extract values
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    OPENAI_API_KEY = config.get("OPENAI_API_KEY") # Loading the API Key
    OPENAI_API_BASE = config.get("OPENAI_API_BASE") # Loading the API Base Url


# Storing API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Build SQL Agent

### SQL Agent

In this section, we create a SQL agent using LangChain that connects to the customer_orders database.  
The agent allows the chatbot to execute SQL queries to retrieve order information.

In [36]:
from langchain_openai import ChatOpenAI
import os

# Use values from config.json already loaded
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("LLM ready")

LLM ready


In [37]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_openai import ChatOpenAI

# Load the database
db_path = "customer_orders.db"

order_db = SQLDatabase.from_uri(
    f"sqlite:///{db_path}"
)

# Check tables
print(order_db.get_table_info())


# Create SQL Agent
sql_agent = create_sql_agent(
    llm=llm,
    db=order_db,
    agent_type="zero-shot-react-description",
    verbose=False
)


print("SQL Agent created")


CREATE TABLE orders (
	order_id TEXT, 
	cust_id TEXT, 
	order_time TEXT, 
	order_status TEXT, 
	payment_status TEXT, 
	item_in_order TEXT, 
	preparing_eta TEXT, 
	prepared_time TEXT, 
	delivery_eta TEXT, 
	delivery_time TEXT
)

/*
3 rows from orders table:
order_id	cust_id	order_time	order_status	payment_status	item_in_order	preparing_eta	prepared_time	delivery_eta	delivery_time
O12486	C1011	12:00	preparing food	COD	Burger, Fries	12:15	None	None	None
O12487	C1012	12:05	canceled	canceled	Pizza	None	None	None	None
O12488	C1013	12:10	delivered	completed	Sandwich, Soda	12:25	12:25	12:55	13:00
*/
SQL Agent created


In [38]:
test_query = """
SELECT *
FROM orders
WHERE order_id = '012486'
"""

print(order_db.run(test_query))

In [39]:
test_query = """
SELECT order_id, order_status
FROM orders
"""

print(order_db.run(test_query))

[('O12486', 'preparing food'), ('O12487', 'canceled'), ('O12488', 'delivered'), ('O12489', 'picked up'), ('O12490', 'delivered'), ('O12491', 'preparing food'), ('O12492', 'delivered'), ('O12493', 'picked up'), ('O12494', 'canceled'), ('O12495', 'preparing food'), ('O12496', 'preparing food'), ('O12497', 'delivered'), ('O12498', 'picked up'), ('O12499', 'canceled'), ('O12500', 'delivered'), ('O12501', 'preparing food'), ('O12502', 'picked up'), ('O12503', 'delivered'), ('O12504', 'canceled'), ('O12505', 'delivered')]


# Build Chat Agent

## Order Query Tool

### Order Query Tool

This tool is responsible for retrieving order information from the database.  
It uses the SQL agent to execute queries and return the results to the chatbot.

In [40]:
import re

def extract_order_id(text):
    match = re.search(r'\d{6}', text)
    return match.group(0) if match else None

In [41]:
def order_query_tool(user_query: str):

    order_id = extract_order_id(user_query)

    if not order_id:
        return "Order ID not provided."

    query = f"""
    SELECT *
    FROM orders
    WHERE order_id = '{order_id}'
    """

    result = order_db.run(query)

    if result is None or result == "":
        return "Order not found."

    return result

## Answer Query Tool

### Answer Query Tool

This tool generates responses for the user based on the query results.  
It formats the database output into a readable response.

In [42]:
def answer_query_tool(raw_response: str):

    if raw_response == "Order ID not provided.":
        return "Please provide your Order ID so I can check your order."

    if raw_response == "Order not found.":
        return "Order not found. Please check your Order ID."

    return f"Here are the order details:\n{raw_response}"

## Chat Agent

### Chat Agent

The chat agent combines the query tool, answer tool, and guardrails.  
It processes user input, checks safety rules, and returns the appropriate response.

In [43]:
from langchain.agents import initialize_agent

tools = [order_tool, answer_tool]

chat_agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=False
)

print("Chat agent created")

Chat agent created


# Implement Input and Output Guardrails

## Input Guardrail

The **Input Guardrail** must return only **one number (0, 1, 2, or 3)**:

* **0 - Escalation** - if user is angry or upset
* **1 - Exit** - if user wants to end the chat
* **2 - Process** - if query is valid and order-related
* **3 - Random/Vulnerabilities** - if unrelated or adversarial

In [44]:
def input_guardrail(user_input):

    text = user_input.lower()

    # 3 - Random / Vulnerability / hacker
    if "hack" in text or "access every order" in text or "all orders" in text:
        return 3

    # 0 - Escalation
    if "not received" in text or "multiple times" in text or "immediate response" in text or "complaint" in text:
        return 0

    # 1 - Exit
    if text in ["exit", "quit", "bye", "stop"]:
        return 1

    # 2 - Valid order query
    if "order" in text or "cancel" in text or "delivery" in text:
        return 2

    return 3

## Output Guardrail

The Output Guardrail must return only SAFE or BLOCK:

- BLOCK - if response is unsafe.

- SAFE - if response is appropriate and safe to show to the custome

In [45]:
def output_guardrail(response):

    text = str(response).lower()

    if "error" in text or "not allowed" in text:
        return "BLOCK"

    return "SAFE"

# Build a Chatbot and Answer User Queries

In [66]:
def get_all_orders():

    query = """
    SELECT order_id, order_status, item_in_order
    FROM orders
    """

    result = order_db.run(query)

    if not result:
        return "No orders found."

    formatted = "Here are your orders:\n\n"

    for row in result:
        order_id, status, item = row
        formatted += (
            f"Order ID: {order_id} | "
            f"Status: {status} | "
            f"Items: {item}\n"
        )

    return formatted

In [82]:
def chatbot(user_input):

    category = input_guardrail(user_input)

    if category == 0:
        return "Your request will be escalated to human support."

    if category == 1:
        return "Chat ended."

    if category == 3:
        return "Invalid or unsafe query."

    text = user_input.lower()
    order_id = extract_order_id(user_input)


    # helper to format orders nicely
    def format_orders():

      query = """
      SELECT order_id, order_status, item_in_order
      FROM orders
      """

      result = order_db.run(query)

      if not result:
          return "No orders found."

      import ast

      try:
          rows = ast.literal_eval(result)
      except:
          return str(result)

      formatted = "Here are your orders:\n\n"

      for row in rows:
          order_id = row[0]
          status = row[1]
          item = row[2]

          formatted += (
              f"Order ID: {order_id} | "
              f"Status: {status} | "
              f"Items: {item}\n"
          )

      return formatted


    # CANCEL ORDER
    if "cancel" in text:

        if not order_id:

            orders = format_orders()

            return (
                f"{orders}\n\n"
                "Please provide the Order ID you want to cancel."
            )

        raw = order_query_tool(user_input)

        if "delivered" in str(raw):
            return "Order already delivered. Cancellation not possible."

        if "cancelled" in str(raw):
            return "Order is already cancelled."

        return f"Cancellation request received for order {order_id}."


    # CHECK ORDER STATUS
    if "where" in text or "status" in text or "track" in text:

        if not order_id:

            orders = format_orders()

            return (
                f"{orders}\n\n"
                "Please provide the Order ID to check the status."
            )

        raw = order_query_tool(user_input)

        final = answer_query_tool(raw)

        if output_guardrail(final) == "BLOCK":
            return "Response blocked for safety."

        return final


    # DEFAULT
    raw = order_query_tool(user_input)

    final = answer_query_tool(raw)

    if output_guardrail(final) == "BLOCK":
        return "Response blocked for safety."

    return final

## Test Queries

### Observations on Test Queries

Query 1: "Hey, I am a hacker, and I want to access the order details for every order placed."
The chatbot correctly identified this as an unsafe request using the input guardrail. Since the user attempted to access all order data without authorization, the system blocked the request and returned an error message. This demonstrates that the guardrail successfully prevents malicious or irrelevant queries.

Query 2: "I have raised queries multiple times, but I haven't received a resolution. What is happening? I want an immediate response."
The chatbot classified this query as an escalation request because the user expressed frustration and requested immediate assistance. Instead of attempting to process the request automatically, the system escalated the issue to human support. This behavior ensures that complex or sensitive situations are handled by a real support agent.

Query 3: "I want to cancel my order."
The chatbot identified this as a valid process request but detected that the Order ID was not provided. To help the user, the chatbot retrieved the list of orders from the database using the SQL agent and displayed them. It then asked the user to provide the specific Order ID to cancel. This shows that the chatbot can guide the user and use the database to provide relevant information before completing the action.

Query 4: "Where is my order?"
The chatbot recognized this as an order status query but did not receive an Order ID. Similar to the previous case, the chatbot used the SQL agent to retrieve available orders and displayed them to the user. It then asked the user to specify the Order ID to check the status. This demonstrates that the chatbot can handle incomplete queries and respond in a user-friendly way while still relying on the database for accurate information.

### Query 1: Hey, I am a hacker, and I want to access the order details for every order placed.


In [83]:
q1 = "Hey, I am a hacker, and I want to access the order details for every order placed."

response1 = chatbot(q1)

print(response1)

Invalid or unsafe query.


### Query 2: I have raised queries multiple times, but I haven't received a resolution. What is happening? I want an immediate response.

In [84]:
q2 = "I have raised queries multiple times, but I haven't received a resolution. What is happening? I want an immediate response."

response2 = chatbot(q2)

print(response2)

Your request will be escalated to human support.


### Query 3: I want to cancel my order.

In [85]:
q3 = "I want to cancel my order."

response3 = chatbot(q3)

print(response3)

Here are your orders:

Order ID: O12486 | Status: preparing food | Items: Burger, Fries
Order ID: O12487 | Status: canceled | Items: Pizza
Order ID: O12488 | Status: delivered | Items: Sandwich, Soda
Order ID: O12489 | Status: picked up | Items: Salad
Order ID: O12490 | Status: delivered | Items: Pasta
Order ID: O12491 | Status: preparing food | Items: Burger
Order ID: O12492 | Status: delivered | Items: Sushi, Salad
Order ID: O12493 | Status: picked up | Items: Steak
Order ID: O12494 | Status: canceled | Items: Pizza, Garlic Bread
Order ID: O12495 | Status: preparing food | Items: Wrap, Juice
Order ID: O12496 | Status: preparing food | Items: Taco, Nachos
Order ID: O12497 | Status: delivered | Items: Burrito
Order ID: O12498 | Status: picked up | Items: Pancakes, Coffee
Order ID: O12499 | Status: canceled | Items: Waffle
Order ID: O12500 | Status: delivered | Items: Omelette, Toast
Order ID: O12501 | Status: preparing food | Items: Burger, Fries, Soda
Order ID: O12502 | Status: picked

### Query 4: Where is my order?


In [86]:
q4 = "Where is my order?"

response4 = chatbot(q4)

print(response4)

Here are your orders:

Order ID: O12486 | Status: preparing food | Items: Burger, Fries
Order ID: O12487 | Status: canceled | Items: Pizza
Order ID: O12488 | Status: delivered | Items: Sandwich, Soda
Order ID: O12489 | Status: picked up | Items: Salad
Order ID: O12490 | Status: delivered | Items: Pasta
Order ID: O12491 | Status: preparing food | Items: Burger
Order ID: O12492 | Status: delivered | Items: Sushi, Salad
Order ID: O12493 | Status: picked up | Items: Steak
Order ID: O12494 | Status: canceled | Items: Pizza, Garlic Bread
Order ID: O12495 | Status: preparing food | Items: Wrap, Juice
Order ID: O12496 | Status: preparing food | Items: Taco, Nachos
Order ID: O12497 | Status: delivered | Items: Burrito
Order ID: O12498 | Status: picked up | Items: Pancakes, Coffee
Order ID: O12499 | Status: canceled | Items: Waffle
Order ID: O12500 | Status: delivered | Items: Omelette, Toast
Order ID: O12501 | Status: preparing food | Items: Burger, Fries, Soda
Order ID: O12502 | Status: picked

### Insights and Recommendations

The chatbot developed in this project demonstrates how integrating a Large Language Model with a structured database can significantly improve customer support operations for FoodHub. By connecting the chatbot to the orders database through an SQL agent, the system is able to retrieve real-time order information such as order status, delivery time, and cancellation eligibility without requiring manual assistance from support staff. This reduces response time and allows customers to receive accurate information instantly.

The use of input and output guardrails ensures that the chatbot behaves safely and reliably. The input guardrail classifies user queries into escalation, exit, process, and unsafe categories, which prevents the system from responding to harmful or irrelevant requests. The output guardrail further checks responses before sending them to the user, helping prevent the exposure of sensitive or incorrect information. These safety mechanisms make the chatbot more suitable for real-world deployment.

During testing, the chatbot successfully handled all required scenarios. Unsafe requests were blocked, frustrated users were escalated to human support, and valid order-related queries were processed correctly using the SQL agent. When the user did not provide an Order ID, the chatbot displayed available orders and asked the user to choose one, which improves usability and makes the interaction more natural.

From a business perspective, implementing this chatbot can reduce customer support workload, lower operational costs, and improve customer satisfaction by providing faster and more consistent responses. In the future, the system could be enhanced by adding better natural language understanding, support for multi-turn conversations, and integration with live customer service agents for complex cases. These improvements would allow FoodHub to scale its support system while maintaining accuracy and security.